<a href="https://colab.research.google.com/github/MiniRedTrout/Segmentation/blob/main/2dTo3dNii.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import cv2
import os
import re
from tqdm import tqdm
import nibabel as nib

In [14]:
def combine2d_to_3d(pred_dir, output_path):
    voxel_size = (1.0,1.0,1.0,1.0)
    pred_files = [f for f in os.listdir(pred_dir) if f.startswith('pred_')]
    def slice_number(flnm):
      m = re.search(r'pred_(\d+)_(\d+)_image', flnm)
      if m:
        return int(m.group(2))
      else:
        return 0
    pred_files = sorted(pred_files,key=slice_number)
    img = cv2.imread(os.path.join(pred_dir, pred_files[0]),cv2.IMREAD_GRAYSCALE)
    h,w = img.shape
    volume = np.zeros((h,w,len(pred_files)),dtype=np.uint8)
    for i, flnm in enumerate(tqdm(pred_files,desc='В 3D')):
      img_path = os.path.join(pred_dir,flnm)
      img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
      volume[:,:,i] = img
    affine = np.eye(4)
    affine[0,0] = voxel_size[0]
    affine[1,1] = voxel_size[1]
    affine[2,2] = voxel_size[2]
    image = nib.Nifti1Image(volume,affine)
    image.header['descrip'] = '3D segmentation from 2D predictions'
    image.header['cal_min'] = volume.min()
    image.header['cal_max'] = volume.max()

    nib.save(image, output_path)
    return image

In [16]:
combine2d_to_3d('/content/drive/MyDrive/Folder1', '/content/drive/MyDrive/Fold3')

В 3D: 100%|██████████| 26/26 [00:08<00:00,  3.11it/s]
